# Maharashtra Joint Crop Disease Training
This notebook trains the upgraded joint crop+disease model for Smart Agri Assistant.

Goal: improve disease and crop recognition for Maharashtra-focused crops using PlantVillage plus local datasets.

## Setup

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
from src.joint_disease_model import JointDiseaseTrainingConfig, train_joint_disease_classifier
from src.disease_data import collect_plant_disease_samples
ROOT


## Choose datasets

In [ ]:
dataset_dirs = [
    ROOT / 'data' / 'external' / 'plantvillage dataset' / 'color',
    # ROOT / 'data' / 'external' / 'maharashtra_leaf_dataset',
    # Path(r'D:/Major Project sem 6 spit/Banana Disease Recognition Dataset/Original Images/Original Images'),
    # Path(r'D:/Major Project sem 6 spit/Cotton Disease/train'),
    # Path(r'D:/Major Project sem 6 spit/Rice_Leaf_AUG'),
    # Path(r'D:/Major Project sem 6 spit/rice_leaf_diseases'),
]
dataset_dirs = [path for path in dataset_dirs if path.exists()]
dataset_dirs


## Inspect class coverage

In [ ]:
samples = collect_plant_disease_samples(dataset_dirs)
print('Images:', len(samples))
print('Pair classes:', samples['class_name'].nunique())
print('Crops:', samples['crop_name'].nunique())
samples[['crop_name', 'disease_name']].head()


## Train the model

In [ ]:
config = JointDiseaseTrainingConfig(
    dataset_dirs=dataset_dirs,
    backbone='densenet121',
    image_size=224,
    batch_size=16,
    head_epochs=10,
    fine_tune_epochs=24,
    learning_rate=8e-4,
    fine_tune_learning_rate=6e-5,
    crop_loss_weight=0.25,
    class_loss_weight=1.0,
    fine_tune_layers=100,
)
metrics = train_joint_disease_classifier(config)
print(json.dumps({
    'pair_accuracy': metrics['pair_accuracy'],
    'crop_accuracy': metrics['crop_accuracy'],
    'disease_accuracy_from_pair_head': metrics['disease_accuracy_from_pair_head'],
    'pair_top3_accuracy': metrics['pair_top3_accuracy'],
}, indent=2))


## Review saved outputs

In [ ]:
report_dir = ROOT / 'reports' / 'disease_joint'
list(report_dir.iterdir()) if report_dir.exists() else 'No reports yet'
